In [1]:
import sys
from pathlib import Path
import torch
import numpy as np

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
if (REPO_ROOT / "src").exists() is False:
    REPO_ROOT = NOTEBOOK_DIR.parent.parent

sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /home/oskar/work/DeepPeptide


In [4]:
EMB_DIR = REPO_ROOT / "data" / "embeddings_esm2"              
DATA_FILE = REPO_ROOT / "data" / "labeled_sequences.csv" 
PART_FILE = REPO_ROOT / "data" / "graphpart_assignments.csv"

print("Embeddings:", EMB_DIR)
print("Data file:", DATA_FILE)
print("Partitions:", PART_FILE)
print("Emb files:", len(list(EMB_DIR.glob("*.pt"))))

Embeddings: /home/oskar/work/DeepPeptide/data/embeddings_esm2
Data file: /home/oskar/work/DeepPeptide/data/labeled_sequences.csv
Partitions: /home/oskar/work/DeepPeptide/data/graphpart_assignments.csv
Emb files: 7931


In [5]:
from src.utils.dataset import PrecomputedCSVForOverlapCRFDataset

ds = PrecomputedCSVForOverlapCRFDataset(
    embeddings_dir=str(EMB_DIR),
    data_file=str(DATA_FILE),
    partitioning_file=str(PART_FILE),
    partitions=[0,1,2],  
    label_type="multistate_with_propeptides", 
    device="cpu"
)

print("len(ds):", len(ds))
print("columns:", list(ds.data.columns))
print(ds.data.head(3)[["sequence","coordinates","propeptide_coordinates","cluster"]].fillna("").to_string())

len(ds): 4372
columns: ['Unnamed: 0', 'protein_name', 'sequence', 'organism', 'is_peptide', 'coordinates', 'is_propeptide', 'propeptide_coordinates', 'priority', 'label-val', 'between_connectivity', 'cluster', 'hash', 'true_peptides', 'true_propeptides']
                                                                                                                                                                                                                                                                                sequence                                                                                                   coordinates propeptide_coordinates  cluster
protein_id                                                                                                                                                                                                                                                                                                                        

/home/oskar/work/DeepPeptide/src/utils/dataset.py:487: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.fillna('') # empty coordinates would become nan.


In [6]:
len(ds.data), len(ds.data[ds.data['organism'] == 'Homo sapiens (Human)'])

(4372, 281)

In [7]:
for i in range(6):
    print(ds.data.is_peptide[i])
    print(ds.data.is_propeptide[i])
    print()

00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000011111001111100111111111111111111110011111001111111111111111111111111111111111111111001111111100000000000000001111100000000000000011111001111111111111111111111111111111
00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000111111111111000000000111111111110000000000000000000000000000000000000000

000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000111110011111001111111111111111111100111110011111111111111111111111111111111111111111001111111100000000000000001111100000000000000011111001111111111111111111111001111111
00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000011111111

/tmp/ipykernel_11719/2341380029.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(ds.data.is_peptide[i])
/tmp/ipykernel_11719/2341380029.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(ds.data.is_propeptide[i])


In [8]:
i = 0
emb, mask, label, peptides = ds[i]

print("idx:", i)
print("emb shape:", tuple(emb.shape), "dtype:", emb.dtype)      # [L, D] или [L, 1280]
print("mask shape:", tuple(mask.shape), "dtype:", mask.dtype)   # [L]
print("label shape:", tuple(label.shape), "dtype:", label.dtype)# [L] int
print("seq_len from ds:", len(ds.sequences[i]))
print("mask min/max:", mask.min().item(), mask.max().item())
print("label min/max:", int(label.min()), int(label.max()))
print("peptides:", peptides)

L_emb = emb.shape[0]
L_seq = len(ds.sequences[i])
print("L_emb == L_seq ?", L_emb == L_seq, " (L_emb:", L_emb, "L_seq:", L_seq, ")")

idx: 0
emb shape: (263, 1280) dtype: torch.float32
mask shape: (263,) dtype: torch.float32
label shape: (263,) dtype: torch.int64
seq_len from ds: 263
mask min/max: 1.0 1.0
label min/max: 0 100
peptides: ([(97, 101), (104, 108), (111, 130), (133, 137), (140, 179), (182, 189), (206, 210), (226, 230), (233, 261), (233, 254), (257, 263)], [(192, 203), (213, 223)])
L_emb == L_seq ? True  (L_emb: 263 L_seq: 263 )


In [9]:
lengths = []
bad = []
for idx in range(min(len(ds), 2000)):  
    emb, mask, label, peptides = ds[idx]
    L_emb = emb.shape[0]
    L_seq = len(ds.sequences[idx])
    lengths.append(L_seq)
    if L_emb != L_seq:
        bad.append((idx, L_seq, L_emb, ds.names[idx]))

print("checked:", len(lengths))
print("len stats: min/median/max =", np.min(lengths), int(np.median(lengths)), np.max(lengths))
print("mismatches:", len(bad))
bad[:10]

checked: 2000
len stats: min/median/max = 8 164 1023
mismatches: 0


[]

In [10]:
from torch.utils.data import DataLoader

loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=ds.collate_fn, num_workers=0)
embB, maskB, labB, pepB = next(iter(loader))

print("batch embeddings:", embB.shape, embB.dtype)  # [B, D, Lmax]
print("batch mask:", maskB.shape, maskB.dtype)      # [B, Lmax]
print("batch labels:", labB.shape, labB.dtype)      # [B, Lmax]
print("mask unique:", torch.unique(maskB))
print("labels min/max:", int(labB.min()), int(labB.max()))
print("peptides[0]:", pepB[0])


batch embeddings: torch.Size([4, 1280, 233]) torch.float32
batch mask: torch.Size([4, 233]) torch.float32
batch labels: torch.Size([4, 233]) torch.int64
mask unique: tensor([0., 1.])
labels min/max: 0 100
peptides[0]: ([], [(77, 107)])
